In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ============================================
# DỮ LIỆU MẪU (15 DÒNG)
# ============================================
data = {
    'Pregnancies': [6, 1, 8, 1, 0, 5, 3, 10, 2, 8, 4, 10, 10, 1, 5],
    'Glucose': [148, 85, 183, 89, 137, 116, 78, 115, 197, 125, 110, 168, 139, 189, 166],
    'BloodPressure': [72, 66, 64, 66, 40, 74, 50, 0, 70, 96, 92, 74, 80, 60, 72],
    'SkinThickness': [35, 29, 0, 23, 35, 0, 32, 0, 45, 0, 0, 0, 0, 23, 19],
    'Insulin': [0, 0, 0, 94, 168, 0, 88, 0, 543, 0, 0, 0, 0, 846, 175],
    'BMI': [33.6, 26.6, 23.3, 28.1, 43.1, 25.6, 31.0, 35.3, 30.5, 0.0, 37.6, 38.0, 27.1, 30.1, 25.8],
    'DiabetesPedigreeFunction': [0.627, 0.351, 0.672, 0.167, 2.288, 0.201, 0.248, 0.134, 0.158, 0.232, 0.191, 0.537, 1.441, 0.398, 0.587],
    'Age': [50, 31, 32, 21, 33, 30, 26, 29, 53, 54, 30, 34, 57, 59, 51],
    'Outcome': [1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1]
}

df = pd.DataFrame(data)

print("="*80)
print("🏥 NAIVE BAYES - XỬ LÝ MISSING VALUES CHÍNH XÁC")
print("="*80)

print(f"\n📊 Dữ liệu gốc (5 dòng đầu):")
print(df.head())
print(f"\n   Tổng số mẫu: {len(df)}")
print(f"   - Có tiểu đường: {(df['Outcome']==1).sum()} người")
print(f"   - Không tiểu đường: {(df['Outcome']==0).sum()} người")

🏥 NAIVE BAYES - XỬ LÝ MISSING VALUES CHÍNH XÁC

📊 Dữ liệu gốc (5 dòng đầu):
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

   Tổng số mẫu: 15
   - Có tiểu đường: 9 người
   - Không tiểu đường: 6 người


In [3]:
# ============================================
# BƯỚC 1: PHÁT HIỆN VÀ PHÂN TÍCH MISSING VALUES
# ============================================
print("\n" + "="*80)
print("📋 BƯỚC 1: PHÁT HIỆN MISSING VALUES (Giá trị = 0)")
print("="*80)

# Các cột có thể có giá trị 0 không hợp lệ
missing_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print("\n⚠️  Phân tích giá trị 0 (MISSING):\n")
print(f"{'Thuộc tính':<20} {'Số lượng 0':<15} {'Tỷ lệ %':<15} {'Trạng thái'}")
print("-" * 70)

missing_summary = {}
for col in missing_cols:
    zero_count = (df[col] == 0).sum()
    zero_pct = (zero_count / len(df)) * 100
    missing_summary[col] = {'count': zero_count, 'pct': zero_pct}

    status = "❌ CÓ MISSING" if zero_count > 0 else "✅ OK"
    print(f"{col:<20} {zero_count:<15} {zero_pct:<14.1f}% {status}")

print("\n💡 LƯU Ý:")
print("   - Glucose = 0: KHÔNG thể sống (missing)")
print("   - BloodPressure = 0: KHÔNG hợp lệ (missing)")
print("   - SkinThickness = 0: Không đo được (missing)")
print("   - Insulin = 0: Không đo được (missing)")
print("   - BMI = 0: KHÔNG hợp lệ (missing)")


📋 BƯỚC 1: PHÁT HIỆN MISSING VALUES (Giá trị = 0)

⚠️  Phân tích giá trị 0 (MISSING):

Thuộc tính           Số lượng 0      Tỷ lệ %         Trạng thái
----------------------------------------------------------------------
Glucose              0               0.0           % ✅ OK
BloodPressure        1               6.7           % ❌ CÓ MISSING
SkinThickness        7               46.7          % ❌ CÓ MISSING
Insulin              9               60.0          % ❌ CÓ MISSING
BMI                  1               6.7           % ❌ CÓ MISSING

💡 LƯU Ý:
   - Glucose = 0: KHÔNG thể sống (missing)
   - BloodPressure = 0: KHÔNG hợp lệ (missing)
   - SkinThickness = 0: Không đo được (missing)
   - Insulin = 0: Không đo được (missing)
   - BMI = 0: KHÔNG hợp lệ (missing)


In [4]:
# ============================================
# BƯỚC 2: THAY THẾ MISSING BẰNG MEDIAN
# ============================================
print("\n" + "="*80)
print("📋 BƯỚC 2: THAY THẾ MISSING BẰNG MEDIAN")
print("="*80)

df_processed = df.copy()

print("\n🔧 Chiến lược: Thay thế giá trị 0 → Median của các giá trị > 0\n")
print(f"{'Thuộc tính':<20} {'Median':<15} {'Số giá trị thay thế'}")
print("-" * 55)

median_values = {}
for col in missing_cols:
    # Tính median từ các giá trị > 0
    valid_values = df[df[col] > 0][col]

    if len(valid_values) > 0:
        median_val = valid_values.median()
        median_values[col] = median_val

        # Thay thế 0 bằng median
        zero_mask = df_processed[col] == 0
        zero_count = zero_mask.sum()
        df_processed.loc[zero_mask, col] = median_val

        print(f"{col:<20} {median_val:<14.1f} {zero_count}")
    else:
        print(f"{col:<20} {'N/A':<14} 0 (không có giá trị hợp lệ)")

print("\n✅ Đã thay thế tất cả missing values bằng median!")

# Hiển thị trước và sau
print("\n📊 SO SÁNH TRƯỚC VÀ SAU XỬ LÝ:\n")
print("Dòng 8 (BloodPressure có 0):")
print(f"   Trước: BloodPressure = {df.iloc[7]['BloodPressure']}")
print(f"   Sau:   BloodPressure = {df_processed.iloc[7]['BloodPressure']:.1f}")

print("\nDòng 10 (BMI có 0):")
print(f"   Trước: BMI = {df.iloc[9]['BMI']}")
print(f"   Sau:   BMI = {df_processed.iloc[9]['BMI']:.1f}")

# Kiểm tra không còn giá trị 0
print("\n🔍 Kiểm tra sau xử lý:")
for col in missing_cols:
    zero_count = (df_processed[col] == 0).sum()
    if zero_count == 0:
        print(f"   ✅ {col}: Không còn giá trị 0")
    else:
        print(f"   ⚠️  {col}: Còn {zero_count} giá trị 0")


📋 BƯỚC 2: THAY THẾ MISSING BẰNG MEDIAN

🔧 Chiến lược: Thay thế giá trị 0 → Median của các giá trị > 0

Thuộc tính           Median          Số giá trị thay thế
-------------------------------------------------------
Glucose              137.0          0
BloodPressure        71.0           1
SkinThickness        30.5           7
Insulin              171.5          9
BMI                  30.3           1

✅ Đã thay thế tất cả missing values bằng median!

📊 SO SÁNH TRƯỚC VÀ SAU XỬ LÝ:

Dòng 8 (BloodPressure có 0):
   Trước: BloodPressure = 0.0
   Sau:   BloodPressure = 71.0

Dòng 10 (BMI có 0):
   Trước: BMI = 0.0
   Sau:   BMI = 30.3

🔍 Kiểm tra sau xử lý:
   ✅ Glucose: Không còn giá trị 0
   ✅ BloodPressure: Không còn giá trị 0
   ✅ SkinThickness: Không còn giá trị 0
   ✅ Insulin: Không còn giá trị 0
   ✅ BMI: Không còn giá trị 0


In [5]:
# ============================================
# BƯỚC 3: RỜI RẠC HÓA
# ============================================
print("\n" + "="*80)
print("📋 BƯỚC 3: RỜI RẠC HÓA DỮ LIỆU")
print("="*80)

# Hàm rời rạc hóa
def discretize_pregnancies(x):
    if x == 0:
        return 'Chưa có'
    elif x <= 3:
        return 'Ít'
    elif x <= 6:
        return 'TB'
    else:
        return 'Nhiều'

def discretize_glucose(x):
    if x < 100:
        return 'Thấp'
    elif x <= 140:
        return 'TB'
    else:
        return 'Cao'

def discretize_bp(x):
    if x < 70:
        return 'Thấp'
    elif x <= 90:
        return 'TB'
    else:
        return 'Cao'

def discretize_skin(x):
    if x <= 20:
        return 'Mỏng'
    elif x <= 30:
        return 'TB'
    else:
        return 'Dày'

def discretize_insulin(x):
    if x <= 100:
        return 'Thấp'
    elif x <= 200:
        return 'TB'
    else:
        return 'Cao'

def discretize_bmi(x):
    if x < 18.5:
        return 'Thiếu cân'
    elif x < 25:
        return 'BT'
    elif x < 30:
        return 'Thừa cân'
    else:
        return 'Béo phì'

def discretize_dpf(x):
    if x < 0.3:
        return 'Thấp'
    elif x <= 0.6:
        return 'TB'
    else:
        return 'Cao'

def discretize_age(x):
    if x < 30:
        return 'Trẻ'
    elif x <= 50:
        return 'TN'
    else:
        return 'Cao'

# Áp dụng rời rạc hóa
df_processed['Pregnancies_Level'] = df_processed['Pregnancies'].apply(discretize_pregnancies)
df_processed['Glucose_Level'] = df_processed['Glucose'].apply(discretize_glucose)
df_processed['BP_Level'] = df_processed['BloodPressure'].apply(discretize_bp)
df_processed['Skin_Level'] = df_processed['SkinThickness'].apply(discretize_skin)
df_processed['Insulin_Level'] = df_processed['Insulin'].apply(discretize_insulin)
df_processed['BMI_Level'] = df_processed['BMI'].apply(discretize_bmi)
df_processed['DPF_Level'] = df_processed['DiabetesPedigreeFunction'].apply(discretize_dpf)
df_processed['Age_Level'] = df_processed['Age'].apply(discretize_age)

print("\n✅ Đã rời rạc hóa 8 thuộc tính")

print("\n📊 Dữ liệu sau tiền xử lý (5 dòng đầu):")
display_cols = ['Glucose', 'Glucose_Level', 'BMI', 'BMI_Level', 'Age', 'Age_Level', 'Outcome']
print(df_processed[display_cols].head())


📋 BƯỚC 3: RỜI RẠC HÓA DỮ LIỆU

✅ Đã rời rạc hóa 8 thuộc tính

📊 Dữ liệu sau tiền xử lý (5 dòng đầu):
   Glucose Glucose_Level   BMI BMI_Level  Age Age_Level  Outcome
0      148           Cao  33.6   Béo phì   50        TN        1
1       85          Thấp  26.6  Thừa cân   31        TN        0
2      183           Cao  23.3        BT   32        TN        1
3       89          Thấp  28.1  Thừa cân   21       Trẻ        0
4      137            TB  43.1   Béo phì   33        TN        1


In [6]:
# ============================================
# BƯỚC 4: XÂY DỰNG NAIVE BAYES
# ============================================
print("\n" + "="*80)
print("📋 BƯỚC 4: XÂY DỰNG MÔ HÌNH NAIVE BAYES")
print("="*80)

class NaiveBayesClassifier:
    def __init__(self, alpha=1):
        self.alpha = alpha
        self.priors = {}
        self.likelihoods = defaultdict(lambda: defaultdict(dict))
        self.classes = []

    def fit(self, X, y, feature_names):
        self.feature_names = feature_names
        self.classes = sorted(y.unique())
        n_samples = len(y)

        # Tính prior
        print("\n🎯 Prior Probabilities:")
        for c in self.classes:
            count = (y == c).sum()
            self.priors[c] = count / n_samples
            label = "Có tiểu đường" if c == 1 else "Không tiểu đường"
            print(f"   P(Y={c}) = {count}/{n_samples} = {self.priors[c]:.3f} ({label})")

        # Tính likelihood
        print("\n📊 Likelihood với Laplace Smoothing (α=1):")

        for feature in feature_names:
            feature_values = sorted(X[feature].unique())
            n_feature_values = len(feature_values)

            print(f"\n   📌 {feature}:")

            # Tạo bảng likelihood
            for c in self.classes:
                X_c = X[y == c]
                n_c = len(X_c)
                label = "Y=1 (Có)" if c == 1 else "Y=0 (Không)"
                print(f"      {label}:")

                for value in feature_values:
                    count = (X_c[feature] == value).sum()
                    # Laplace smoothing
                    prob = (count + self.alpha) / (n_c + self.alpha * n_feature_values)
                    self.likelihoods[feature][c][value] = prob
                    print(f"         P({feature}={value}|{label}) = ({count}+1)/({n_c}+{n_feature_values}) = {prob:.3f}")

    def predict_proba(self, X_test, verbose=True):
        posteriors = {}

        if verbose:
            print("\n🔮 Tính toán Posterior:")

        for c in self.classes:
            posterior = self.priors[c]

            if verbose:
                label = "Có tiểu đường" if c == 1 else "Không tiểu đường"
                print(f"\n   {label} (Y={c}):")
                print(f"      P(Y={c}) = {self.priors[c]:.3f}")

            for feature in self.feature_names:
                value = X_test[feature]
                likelihood = self.likelihoods[feature][c].get(value, 1/(len(X_test)+self.alpha*10))
                posterior *= likelihood

                if verbose:
                    print(f"      × P({feature}={value}|Y={c}) = {likelihood:.3f}")

            posteriors[c] = posterior

            if verbose:
                print(f"      = {posterior:.6f} (chưa chuẩn hóa)")

        # Normalize
        total = sum(posteriors.values())
        for c in posteriors:
            posteriors[c] /= total

        if verbose:
            print(f"\n   📊 Tổng (để chuẩn hóa): {total:.6f}")

        return posteriors

    def predict(self, X_test, verbose=True):
        posteriors = self.predict_proba(X_test, verbose)
        return max(posteriors, key=posteriors.get)


📋 BƯỚC 4: XÂY DỰNG MÔ HÌNH NAIVE BAYES


In [7]:
# ============================================
# BƯỚC 5: HUẤN LUYỆN VÀ DỰ ĐOÁN
# ============================================
print("\n" + "="*80)
print("📋 BƯỚC 5: HUẤN LUYỆN VÀ DỰ ĐOÁN")
print("="*80)

# Chọn features
selected_features = [
    'Pregnancies_Level',
    'Glucose_Level',
    'BP_Level',
    'Skin_Level',
    'Insulin_Level',
    'BMI_Level',
    'DPF_Level',
    'Age_Level'
]

X_train = df_processed[selected_features]
y_train = df_processed['Outcome']

print(f"\n✅ Sử dụng {len(selected_features)} thuộc tính:")
for i, feat in enumerate(selected_features, 1):
    print(f"   {i}. {feat}")

model = NaiveBayesClassifier(alpha=1)
model.fit(X_train, y_train, selected_features)


📋 BƯỚC 5: HUẤN LUYỆN VÀ DỰ ĐOÁN

✅ Sử dụng 8 thuộc tính:
   1. Pregnancies_Level
   2. Glucose_Level
   3. BP_Level
   4. Skin_Level
   5. Insulin_Level
   6. BMI_Level
   7. DPF_Level
   8. Age_Level

🎯 Prior Probabilities:
   P(Y=0) = 6/15 = 0.400 (Không tiểu đường)
   P(Y=1) = 9/15 = 0.600 (Có tiểu đường)

📊 Likelihood với Laplace Smoothing (α=1):

   📌 Pregnancies_Level:
      Y=0 (Không):
         P(Pregnancies_Level=Chưa có|Y=0 (Không)) = (0+1)/(6+4) = 0.100
         P(Pregnancies_Level=Nhiều|Y=0 (Không)) = (2+1)/(6+4) = 0.300
         P(Pregnancies_Level=TB|Y=0 (Không)) = (2+1)/(6+4) = 0.300
         P(Pregnancies_Level=Ít|Y=0 (Không)) = (2+1)/(6+4) = 0.300
      Y=1 (Có):
         P(Pregnancies_Level=Chưa có|Y=1 (Có)) = (1+1)/(9+4) = 0.154
         P(Pregnancies_Level=Nhiều|Y=1 (Có)) = (3+1)/(9+4) = 0.308
         P(Pregnancies_Level=TB|Y=1 (Có)) = (2+1)/(9+4) = 0.231
         P(Pregnancies_Level=Ít|Y=1 (Có)) = (3+1)/(9+4) = 0.308

   📌 Glucose_Level:
      Y=0 (Không):
      

In [8]:
# ============================================
# BƯỚC 6: DỰ ĐOÁN CHO MẪU TEST
# ============================================
print("\n" + "="*80)
print("📋 BƯỚC 6: DỰ ĐOÁN CHO MẪU TEST")
print("="*80)

# Mẫu test
test_sample = {
    'Pregnancies': 0,
    'Glucose': 118,
    'BloodPressure': 84,
    'SkinThickness': 47,
    'Insulin': 230,
    'BMI': 45.8,
    'DiabetesPedigreeFunction': 0.551,
    'Age': 32
}
# Rời rạc hóa
X_test = {
    'Pregnancies_Level': discretize_pregnancies(test_sample['Pregnancies']),
    'Glucose_Level': discretize_glucose(test_sample['Glucose']),
    'BP_Level': discretize_bp(test_sample['BloodPressure']),
    'Skin_Level': discretize_skin(test_sample['SkinThickness']),
    'Insulin_Level': discretize_insulin(test_sample['Insulin']),
    'BMI_Level': discretize_bmi(test_sample['BMI']),
    'DPF_Level': discretize_dpf(test_sample['DiabetesPedigreeFunction']),
    'Age_Level': discretize_age(test_sample['Age'])
}

print(f"\n📊 Dữ liệu đầu vào:")
for key, value in test_sample.items():
    print(f"   - {key}: {value}")



print(f"\n🔄 Sau rời rạc hóa:")
for key, value in X_test.items():
    print(f"   - {key}: {value}")

# Dự đoán
posteriors = model.predict_proba(X_test)
prediction = model.predict(X_test, verbose=False)

print("\n" + "="*80)
print("🎯 KẾT QUẢ CUỐI CÙNG")
print("="*80)

print(f"\n📈 Xác suất sau chuẩn hóa:")
for c in model.classes:
    label = "Có tiểu đường" if c == 1 else "Không tiểu đường"
    bar = "█" * int(posteriors[c] * 50)
    print(f"   P(Y={c}|X) = {posteriors[c]*100:6.2f}% {bar} ({label})")

print("\n" + "="*80)
if prediction == 1:
    print("⚠️  KẾT LUẬN: CÓ TIỂU ĐƯỜNG")
    print("="*80)
    print("\n💡 Phân tích:")
    print(f"   - Glucose = {test_sample['Glucose']} → {X_test['Glucose_Level']} (nguy cơ)")
    print(f"   - BMI = {test_sample['BMI']} → {X_test['BMI_Level']} (nguy cơ)")
    print(f"   - Age = {test_sample['Age']} → {X_test['Age_Level']}")
else:
    print("✅ KẾT LUẬN: KHÔNG TIỂU ĐƯỜNG")
    print("="*80)
    print("\n💡 Phân tích: Các chỉ số trong mức bình thường")


📋 BƯỚC 6: DỰ ĐOÁN CHO MẪU TEST

📊 Dữ liệu đầu vào:
   - Pregnancies: 0
   - Glucose: 118
   - BloodPressure: 84
   - SkinThickness: 47
   - Insulin: 230
   - BMI: 45.8
   - DiabetesPedigreeFunction: 0.551
   - Age: 32

🔄 Sau rời rạc hóa:
   - Pregnancies_Level: Chưa có
   - Glucose_Level: TB
   - BP_Level: TB
   - Skin_Level: Dày
   - Insulin_Level: Cao
   - BMI_Level: Béo phì
   - DPF_Level: TB
   - Age_Level: TN

🔮 Tính toán Posterior:

   Không tiểu đường (Y=0):
      P(Y=0) = 0.400
      × P(Pregnancies_Level=Chưa có|Y=0) = 0.100
      × P(Glucose_Level=TB|Y=0) = 0.556
      × P(BP_Level=TB|Y=0) = 0.444
      × P(Skin_Level=Dày|Y=0) = 0.556
      × P(Insulin_Level=Cao|Y=0) = 0.111
      × P(BMI_Level=Béo phì|Y=0) = 0.333
      × P(DPF_Level=TB|Y=0) = 0.222
      × P(Age_Level=TN|Y=0) = 0.444
      = 0.000020 (chưa chuẩn hóa)

   Có tiểu đường (Y=1):
      P(Y=1) = 0.600
      × P(Pregnancies_Level=Chưa có|Y=1) = 0.154
      × P(Glucose_Level=TB|Y=1) = 0.250
      × P(BP_Level=TB|Y

In [9]:
# ============================================
# BƯỚC 7: ĐÁNH GIÁ MÔ HÌNH
# ============================================
print("\n" + "="*80)
print("📋 BƯỚC 7: ĐÁNH GIÁ MÔ HÌNH")
print("="*80)

# Dự đoán cho toàn bộ tập training
correct = 0
predictions = []

for idx in range(len(X_train)):
    test_row = X_train.iloc[idx].to_dict()
    pred = model.predict(test_row, verbose=False)
    predictions.append(pred)
    if pred == y_train.iloc[idx]:
        correct += 1

accuracy = correct / len(X_train)

print(f"\n📊 Kết quả trên tập huấn luyện:")
print(f"   - Độ chính xác: {accuracy*100:.2f}%")
print(f"   - Dự đoán đúng: {correct}/{len(X_train)}")

# Ma trận nhầm lẫn
tp = sum([1 for i in range(len(predictions)) if predictions[i] == 1 and y_train.iloc[i] == 1])
tn = sum([1 for i in range(len(predictions)) if predictions[i] == 0 and y_train.iloc[i] == 0])
fp = sum([1 for i in range(len(predictions)) if predictions[i] == 1 and y_train.iloc[i] == 0])
fn = sum([1 for i in range(len(predictions)) if predictions[i] == 0 and y_train.iloc[i] == 1])

print("\n📈 Ma trận nhầm lẫn:")
print(f"""
                Predicted
                0      1
         ┌──────────────┐
    0    │  {tn:2d}     {fp:2d}  │
Actual   │              │
    1    │  {fn:2d}     {tp:2d}  │
         └──────────────┘
""")

if tp + fp > 0:
    precision = tp / (tp + fp)
    print(f"   Precision: {precision*100:.2f}%")
if tp + fn > 0:
    recall = tp / (tp + fn)
    print(f"   Recall: {recall*100:.2f}%")



📋 BƯỚC 7: ĐÁNH GIÁ MÔ HÌNH

📊 Kết quả trên tập huấn luyện:
   - Độ chính xác: 93.33%
   - Dự đoán đúng: 14/15

📈 Ma trận nhầm lẫn:

                Predicted
                0      1
         ┌──────────────┐
    0    │   6      0  │
Actual   │              │
    1    │   1      8  │
         └──────────────┘

   Precision: 100.00%
   Recall: 88.89%
